In [1]:
from __future__ import annotations
import io
import re
import shutil
import subprocess
import tempfile
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
PROJECT_DIR = Path.home() / "Hussein"

WORKDIR = PROJECT_DIR / "mutation_peptides"
OUTPUT_DIR = PROJECT_DIR / "predictions"
DECOY_DIR = PROJECT_DIR / "decoy"

MIXMHCPRED_EXE = (
    Path.home()
    / "Downloads"
    / "MixMHCpred"
    / "MixMHCpred"
)

NETMHCPAN_EXE = (
    Path.home()
    / "Downloads"
    / "netMHCpan-4.2"
    / "netMHCpan"
)

MIXMHC2PRED_EXE = (
    Path.home()
    / "Downloads"
    / "MixMHC2pred-2"
    / "MixMHC2pred"
)

NETMHCIIPAN_EXE = (
    Path.home()
    / "Downloads"
    / "netMHCIIpan-4.3"
    / "netMHCIIpan"
)

MHCFLURRY_EXE = (
    Path.home()
    / "miniconda3"
    / "envs"
    / "mhc-tools"
    / "bin"
    / "mhcflurry-predict"
)

HMMHC_CONDA_ENV = "hmmhc"

RUN_PREDICTIONS = True
SAMPLES = ("ID8", "ID8_CPR")

CLASS_I_ROUTES = {
    8: {"mhcflurry": "H-2-Kb", "mixmhcpred": "H2-Kb", "netmhcpan": "H-2-Kb"},
    9: {"mhcflurry": "H-2-Db", "mixmhcpred": "H2-Db", "netmhcpan": "H-2-Db"},
}
CLASS_II_ROUTES = {
    15: {
        "mixmhc2pred": "H2_Aa_b__H2_Ab_b",
        "netmhciipan": "H-2-IAb",
        "hmmhc": "H2-IAb",
    }
}


PREDICTORS = (
    "mixmhcpred",
    "netmhcpan",
    "mhcflurry",
    "mixmhc2pred",
    "netmhciipan",
    "hmmhc",
)

PREDICTOR_LENGTHS = {
    "mixmhcpred": (8, 9),
    "netmhcpan": (8, 9),
    "mhcflurry": (8, 9),
    "mixmhc2pred": (15,),
    "netmhciipan": (15,),
    "hmmhc": (15,),
}

PREDICTOR_DIRS = {
    "mixmhcpred": "mixmhcpred_class_I",
    "netmhcpan": "netmhcpan_class_I",
    "mhcflurry": "mhcflurry_class_I",
    "mixmhc2pred": "mixmhc2pred_class_II",
    "netmhciipan": "netmhciipan_class_II",
    "hmmhc": "hmmhc_class_II",
}

PREDICTOR_EXTENSIONS = {
    "mixmhcpred": ".output.txt",
    "netmhcpan": ".EL-BA.tsv",
    "mhcflurry": ".output.csv",
    "mixmhc2pred": ".output.txt",
    "netmhciipan": ".xls",
    "hmmhc": ".output.csv",
}
DECOY_DIR = WORKDIR.parent / "decoy"
DECOY_FILES = {
    ("H2-Kb", 8): DECOY_DIR / "decoy-mouse-allele-H2-Kb-length-8.tsv",
    ("H2-Db", 9): DECOY_DIR / "decoy-mouse-allele-H2-Db-length-9.tsv",
    ("I-Ab", 15): DECOY_DIR / "decoy-mouse-allele-I-Ab-length-15.tsv",
}

In [3]:
def read_tsv(path: Path) -> pd.DataFrame:
    path = Path(path)
    table = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
    )
    table.columns = [str(column).strip().lstrip("#") for column in table.columns]
    return table


def normalize_peptides(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.upper()


def route_allele(length: int, predictor: str) -> str:
    routes = CLASS_I_ROUTES if length in CLASS_I_ROUTES else CLASS_II_ROUTES
    return routes[length][predictor]


def add_row_ids(table: pd.DataFrame) -> pd.DataFrame:
    table = table.copy().reset_index(drop=True)
    table["source_row_id"] = [
        f"{sample}|{sequence_type}|{length}|{index}"
        for index, (sample, sequence_type, length) in enumerate(
            table[["sample", "sequence_type", "length"]].itertuples(index=False, name=None)
        )
    ]
    table["prediction_row_id"] = table.groupby(
        ["sample", "sequence_type", "length"], sort=False
    ).cumcount()
    return table

def load_target_input() -> pd.DataFrame:
    frames = []
    for sample in SAMPLES:
        path = WORKDIR / sample / f"{sample}_MT_WT_windows.tsv"
        source = read_tsv(path)
        needed = {"WT_peptide", "MUT_peptide"}
        missing = needed - set(source.columns)
        if missing:
            raise KeyError(f"{path}: missing columns: {sorted(missing)}")
        for sequence_type, column in (("WT", "WT_peptide"), ("MT", "MUT_peptide")):
            frame = pd.DataFrame(
                {
                    "sample": sample,
                    "sequence_type": sequence_type,
                    "peptide": normalize_peptides(source[column]),
                    "window_row": np.arange(len(source), dtype=int),
                }
            )
            frames.append(frame)

    table = pd.concat(frames, ignore_index=True)
    table["length"] = table["peptide"].str.len()
    table = table.loc[table["length"].isin((8, 9, 15))].reset_index(drop=True)
    table = add_row_ids(table)
    return table

In [4]:
def load_decoy_input() -> pd.DataFrame:
    frames = []
    for (allele, expected_length), path in DECOY_FILES.items():
        source = read_tsv(path)
        table = pd.DataFrame(
            {
                "sample": "DECOY",
                "sequence_type": "DECOY",
                "peptide": normalize_peptides(source["peptide"]),
                "allele": allele,
            }
        )
        table["length"] = table["peptide"].str.len()
        frames.append(table)

    result = add_row_ids(pd.concat(frames, ignore_index=True))
    return result


In [5]:
def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(value)).strip("-")


def resolve_executable(executable: str | Path) -> str:
    executable = str(executable)
    if "/" in executable:
        path = Path(executable)
        return str(path)
    resolved = shutil.which(executable)
    return resolved


@contextmanager
def temporary_peptide_txt(peptides: pd.Series):
    path = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as handle:
            path = Path(handle.name)
            for peptide in peptides:
                handle.write(f"{peptide}\n")
        yield path
    finally:
        if path is not None:
            path.unlink(missing_ok=True)


@contextmanager
def temporary_allele_peptide_csv(allele: str, peptides: pd.Series):
    path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".csv", delete=False, newline=""
        ) as handle:
            path = Path(handle.name)
            pd.DataFrame({"allele": allele, "peptide": peptides}).to_csv(handle, index=False)
        yield path
    finally:
        if path is not None:
            path.unlink(missing_ok=True)


def iter_batches(input_table: pd.DataFrame, predictor: str):
    selected = input_table.loc[input_table["length"].isin(PREDICTOR_LENGTHS[predictor])]
    for (sample, sequence_type, length), batch in selected.groupby(
        ["sample", "sequence_type", "length"], sort=True
    ):
        yield sample, sequence_type, int(length), batch.sort_values("prediction_row_id")


def output_path(
    predictor: str,
    dataset_label: str,
    sample: str,
    sequence_type: str,
    length: int,
) -> Path:
    allele = route_allele(length, predictor)
    stem = (
        f"{predictor}-{sample}-{sequence_type}-{safe_name(allele)}-length-{length}"
    )
    folder = OUTPUT_DIR / dataset_label.lower() / PREDICTOR_DIRS[predictor]
    return folder / f"{stem}{PREDICTOR_EXTENSIONS[predictor]}"


def run_command(command: list[str], cwd: Path, log_path: Path) -> None:
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    log_path.write_text(result.stdout + result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {result.returncode}): {' '.join(command)}\n"
            f"--- stdout ---\n{result.stdout}\n--- stderr ---\n{result.stderr}"
        )


def check_output(path: Path) -> None:
    print(f"Created: {path} ({path.stat().st_size:,} bytes)")


def command_for_batch(
    predictor: str,
    allele: str,
    length: int,
    peptides: pd.Series,
    output: Path,
):
    if predictor == "mhcflurry":
        executable = resolve_executable(MHCFLURRY_EXE)
        context = temporary_allele_peptide_csv(allele, peptides)
        input_flag = "csv"
    else:
        executable_paths = {
            "mixmhcpred": MIXMHCPRED_EXE,
            "netmhcpan": NETMHCPAN_EXE,
            "mixmhc2pred": MIXMHC2PRED_EXE,
            "netmhciipan": NETMHCIIPAN_EXE,
        }
        executable = (
            None if predictor == "hmmhc" else resolve_executable(executable_paths[predictor])
        )
        context = temporary_peptide_txt(peptides)
        input_flag = "txt"

    @contextmanager
    def prepared():
        with context as input_path:
            if predictor == "mixmhcpred":
                command = [executable, "-i", str(input_path), "-o", str(output), "-a", allele]
                cwd = Path(executable).parent
            elif predictor == "netmhcpan":
                command = [
                    executable,
                    "-p",
                    "-f",
                    str(input_path),
                    "-a",
                    allele,
                    "-l",
                    str(length),
                    "-BA",
                    "-xls",
                    "-xlsfile",
                    str(output),
                ]
                cwd = Path(executable).parent
            elif predictor == "mhcflurry":
                assert input_flag == "csv"
                command = [
                    executable,
                    str(input_path),
                    "--out",
                    str(output),
                    "--affinity-only",
                    "--no-flanking",
                    "--backend",
                    "cpu",
                    "--no-throw",
                ]
                cwd = output.parent
            elif predictor == "mixmhc2pred":
                command = [
                    executable,
                    "-i",
                    str(input_path),
                    "-o",
                    str(output),
                    "-a",
                    allele,
                    "--no_context",
                ]
                cwd = Path(executable).parent
            elif predictor == "netmhciipan":
                command = [
                    executable,
                    "-f",
                    str(input_path),
                    "-inptype",
                    "1",
                    "-a",
                    allele,
                    "-BA",
                    "-xls",
                    "-xlsfile",
                    str(output),
                ]
                cwd = Path(executable).parent
            elif predictor == "hmmhc":
                command = [
                    "conda",
                    "run",
                    "-n",
                    HMMHC_CONDA_ENV,
                    "hmmhc-predict",
                    "--allele",
                    allele,
                    "--input",
                    str(input_path),
                    "--output",
                    str(output),
                ]
                cwd = output.parent
            else:
                raise KeyError(predictor)
            yield command, cwd

    return prepared()


def run_predictor(
    predictor: str, input_table: pd.DataFrame, dataset_label: str
) -> dict[tuple[str, str, int], Path]:
    outputs = {}
    for sample, sequence_type, length, batch in iter_batches(input_table, predictor):
        allele = route_allele(length, predictor)
        output = output_path(
            predictor, dataset_label, sample, sequence_type, length
        )
        output.parent.mkdir(parents=True, exist_ok=True)
        log = output.with_suffix(output.suffix + ".log")
        print(
            f"Running {predictor}: dataset={dataset_label}, sample={sample}, "
            f"type={sequence_type}, length={length}, allele={allele}, n={len(batch):,}"
        )
        with command_for_batch(
            predictor, allele, length, batch["peptide"], output
        ) as (command, cwd):
            run_command(command, cwd=cwd, log_path=log)
        check_output(output)
        outputs[(sample, sequence_type, length)] = output
    return outputs


def discover_outputs(
    predictor: str, input_table: pd.DataFrame, dataset_label: str
) -> dict[tuple[str, str, int], Path]:
    outputs = {}
    for sample, sequence_type, length, _batch in iter_batches(input_table, predictor):
        path = output_path(predictor, dataset_label, sample, sequence_type, length)
        check_output(path)
        outputs[(sample, sequence_type, length)] = path
    return outputs


def run_all_predictors(
    input_table: pd.DataFrame, dataset_label: str
) -> dict[str, dict[tuple[str, str, int], Path]]:
    action = run_predictor if RUN_PREDICTIONS else discover_outputs
    return {
        predictor: action(predictor, input_table, dataset_label)
        for predictor in PREDICTORS
    }

In [6]:
def parse_mixmhcpred_file(path: Path) -> pd.DataFrame:
    lines = path.read_text().splitlines(keepends=True)
    table = pd.read_csv(io.StringIO("".join(x for x in lines if not x.startswith("#"))), sep="\t")
    table = table.rename(columns={"Peptide": "peptide"})
    score = "Score_bestAllele" if "Score_bestAllele" in table else "Score"
    rank = "%Rank_bestAllele" if "%Rank_bestAllele" in table else "%Rank"
    return table[["peptide", score, rank]].rename(
        columns={score: "mixmhcpred_score", rank: "mixmhcpred_rank"}
    )


def parse_netmhcpan_file(path: Path) -> pd.DataFrame:
    table = pd.read_csv(path, sep="\t", skiprows=2).rename(columns={"Peptide": "peptide"})
    keep, rename = ["peptide"], {}
    if "BA_score" in table:
        keep.append("BA_score")
        rename["BA_score"] = "netmhcpan_ba_score"
    rank = "BA_rank" if "BA_rank" in table else "EL_rank"
    keep.append(rank)
    rename[rank] = "netmhcpan_rank"
    return table[keep].rename(columns=rename)


def parse_mhcflurry_file(path: Path) -> pd.DataFrame:
    table = pd.read_csv(path)
    keep = ["peptide", "mhcflurry_affinity"]
    rename = {}
    if "mhcflurry_affinity_percentile" in table:
        keep.append("mhcflurry_affinity_percentile")
        rename["mhcflurry_affinity_percentile"] = "mhcflurry_rank"
    return table[keep].rename(columns=rename)


def parse_mixmhc2pred_file(path: Path) -> pd.DataFrame:
    lines = path.read_text().splitlines(keepends=True)
    table = pd.read_csv(io.StringIO("".join(x for x in lines if not x.startswith("#"))), sep="\t")
    table = table.rename(columns={"Peptide": "peptide"})
    rank = "%Rank_best"
    keep, rename = ["peptide", rank], {rank: "mixmhc2pred_rank"}
    if "BestAllele" in table:
        keep.append("BestAllele")
        rename["BestAllele"] = "mixmhc2pred_best_allele"
    return table[keep].rename(columns=rename)


def parse_netmhciipan_file(path: Path) -> pd.DataFrame:
    table = pd.read_csv(path, sep="\t", skiprows=2).rename(columns={"Peptide": "peptide"})
    keep, rename = ["peptide"], {}
    if "nM" in table:
        keep.append("nM")
        rename["nM"] = "netmhciipan_affinity_nM"
    rank = "Rank_BA" if "Rank_BA" in table else "Rank_EL"
    keep.append(rank)
    rename[rank] = "netmhciipan_rank"
    return table[keep].rename(columns=rename)


def parse_hmmhc_file(path: Path) -> pd.DataFrame:
    table = pd.read_csv(path)
    required = ["peptide", "-10logOdds"]
    missing = set(required) - set(table.columns)
    if missing:
        raise KeyError(f"{path}: missing hmMHC columns: {sorted(missing)}")
    keep = required.copy()
    rename = {"-10logOdds": "hmmhc_score"}
    if "percentile_rank" in table:
        keep.append("percentile_rank")
        rename["percentile_rank"] = "hmmhc_rank"
    return table[keep].rename(columns=rename)


PARSERS = {
    "mixmhcpred": parse_mixmhcpred_file,
    "netmhcpan": parse_netmhcpan_file,
    "mhcflurry": parse_mhcflurry_file,
    "mixmhc2pred": parse_mixmhc2pred_file,
    "netmhciipan": parse_netmhciipan_file,
    "hmmhc": parse_hmmhc_file,
}

In [7]:
def attach_predictions_by_position(
    input_table: pd.DataFrame,
    outputs: dict[tuple[str, str, int], Path],
    predictor: str,
) -> pd.DataFrame:
    prediction_frames = []
    for (sample, sequence_type, length), path in outputs.items():
        expected = input_table.loc[
            input_table["sample"].eq(sample)
            & input_table["sequence_type"].eq(sequence_type)
            & input_table["length"].eq(length)
        ].sort_values("prediction_row_id")
        parsed = PARSERS[predictor](path).reset_index(drop=True)
        expected_peptides = expected["peptide"].astype(str).str.upper().to_numpy()
        parsed_peptides = parsed["peptide"].astype(str).str.upper().to_numpy()
        parsed = parsed.drop(columns="peptide")
        parsed["sample"] = sample
        parsed["sequence_type"] = sequence_type
        parsed["length"] = length
        parsed["prediction_row_id"] = np.arange(len(parsed), dtype=int)
        prediction_frames.append(parsed)

    combined = pd.concat(prediction_frames, ignore_index=True)
    before = len(input_table)
    result = input_table.merge(
        combined,
        on=["sample", "sequence_type", "length", "prediction_row_id"],
        how="left",
        validate="one_to_one",
    )
    if len(result) != before:
        raise AssertionError(f"{predictor} changed row count")
    return result


def combine_all_predictions(
    input_table: pd.DataFrame,
    output_maps: dict[str, dict[tuple[str, str, int], Path]],
) -> pd.DataFrame:
    result = input_table.copy()
    for predictor in PREDICTORS:
        result = attach_predictions_by_position(result, output_maps[predictor], predictor)
    return result


def save_predictions_by_sample_and_length(
    table: pd.DataFrame, dataset_label: str
) -> list[Path]:
    final_dir = OUTPUT_DIR / "final_tables" / dataset_label.lower()
    final_dir.mkdir(parents=True, exist_ok=True)
    written = []

    for (sample, length), group in table.groupby(["sample", "length"], sort=True):
        path = final_dir / (
            f"{safe_name(sample)}.length-{int(length)}.predictions.tsv"
        )
        group.sort_values(
            ["sequence_type", "prediction_row_id"], kind="stable"
        ).to_csv(path, sep="\t", index=False)
        check_output(path)
        written.append(path)

    expected_groups = table[["sample", "length"]].drop_duplicates().shape[0]
    if len(written) != expected_groups:
        raise AssertionError(
            f"{dataset_label}: wrote {len(written)} final files; "
            f"expected {expected_groups}"
        )
    return written

In [8]:
def make_mutation_key(table: pd.DataFrame) -> pd.Series:
    if "Uploaded_variation" in table:
        return table["Uploaded_variation"].astype(str)
    key = table["Location"].astype(str)
    for column in ("VCF_REF", "VCF_ALT"):
        if column in table:
            key = key + "|" + table[column].astype(str)
    return key


def create_shared_mutation_outputs() -> None:
    missense_files = {
        "ID8": Path(
            "/Users/annaklimova/Hussein/DNA_variant_calling/"
            "STZQGV_1_ID8/STZQGV_1_ID8.unique_location_and_peptide.tsv"
        ),
        "ID8_CPR": Path(
            "/Users/annaklimova/Hussein/DNA_variant_calling/"
            "STZQGV_2_ID8_CPR/STZQGV_2_ID8_CPR.unique_location_and_peptide.tsv"
        ),
    }
    tables = {}
    for sample, path in missense_files.items():
        table = read_tsv(path)
        if "Consequence" in table:
            table = table.loc[
                table["Consequence"].str.contains("missense_variant", na=False)
            ].copy()
        table["mutation_key"] = make_mutation_key(table)
        tables[sample] = table

    id8_keys = set(tables["ID8"]["mutation_key"])
    cpr_keys = set(tables["ID8_CPR"]["mutation_key"])
    shared = id8_keys & cpr_keys
    tables["ID8"].loc[tables["ID8"]["mutation_key"].isin(shared)].to_csv(
        OUTPUT_DIR / "shared_missense_mutations.tsv", sep="\t", index=False
    )
    try:
        from matplotlib_venn import venn2
    except ImportError as exc:
        raise ImportError(
            "Install once with: conda install -c conda-forge matplotlib-venn"
        ) from exc
    figure, axis = plt.subplots(figsize=(7, 6))
    venn2([id8_keys, cpr_keys], set_labels=("ID8", "ID8-CPR"), ax=axis)
    axis.set_title("Shared missense mutations")
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "missense_mutations_venn.png", dpi=300, bbox_inches="tight")
    plt.close(figure)


In [9]:
##  final_tables/
# ├── target/
# │   ├── ID8.length-8.predictions.tsv
# │   ├── ID8.length-9.predictions.tsv
# │   ├── ID8.length-15.predictions.tsv
# │   ├── ID8_CPR.length-8.predictions.tsv
# │   ├── ID8_CPR.length-9.predictions.tsv
# │   └── ID8_CPR.length-15.predictions.tsv
# └── decoy/
#     ├── DECOY.length-8.predictions.tsv
#     ├── DECOY.length-9.predictions.tsv
#     └── DECOY.length-15.predictions.tsv
    

In [10]:
target_input = load_target_input()

target_counts = (
    target_input.groupby(["sample", "sequence_type", "length"])
    .agg(rows=("peptide", "size"), unique_peptides=("peptide", "nunique"))
    .reset_index()
)
print(f"TARGET rows: {len(target_input):,}")
display(target_counts)
display(target_input.head())

TARGET rows: 135,538


,sample,sequence_type,length,rows,unique_peptides
0,ID8,MT,8,8717,8430
1,ID8,MT,9,9781,9474
2,ID8,MT,15,16056,15710
3,ID8,WT,8,8717,8126
4,ID8,WT,9,9781,9122
5,ID8,WT,15,16056,15042
6,ID8_CPR,MT,8,8406,8166
7,ID8_CPR,MT,9,9426,9169
8,ID8_CPR,MT,15,15383,15096
9,ID8_CPR,WT,8,8406,7744


,sample,sequence_type,peptide,window_row,length,source_row_id,prediction_row_id
0,ID8,WT,VKEGKTEK,0,8,ID8|WT|8|0,0
1,ID8,WT,KEGKTEKT,1,8,ID8|WT|8|1,1
2,ID8,WT,EGKTEKTD,2,8,ID8|WT|8|2,2
3,ID8,WT,AVKEGKTEK,3,9,ID8|WT|9|3,0
4,ID8,WT,VKEGKTEKT,4,9,ID8|WT|9|4,1


In [11]:
decoy_input = load_decoy_input()

decoy_counts = (
    decoy_input.groupby(["sample", "allele", "length"])
    .agg(rows=("peptide", "size"), unique_peptides=("peptide", "nunique"))
    .reset_index()
)
print(f"DECOY rows: {len(decoy_input):,}")
display(decoy_counts)
display(decoy_input.head())


DECOY rows: 300,000


,sample,allele,length,rows,unique_peptides
0,DECOY,H2-Db,9,100000,100000
1,DECOY,H2-Kb,8,100000,100000
2,DECOY,I-Ab,15,100000,100000


,sample,sequence_type,peptide,allele,length,source_row_id,prediction_row_id
0,DECOY,DECOY,QAEELMSN,H2-Kb,8,DECOY|DECOY|8|0,0
1,DECOY,DECOY,RASEGVTS,H2-Kb,8,DECOY|DECOY|8|1,1
2,DECOY,DECOY,KDSQLIVL,H2-Kb,8,DECOY|DECOY|8|2,2
3,DECOY,DECOY,RSTSSTPG,H2-Kb,8,DECOY|DECOY|8|3,3
4,DECOY,DECOY,IEEEPAGV,H2-Kb,8,DECOY|DECOY|8|4,4


In [12]:
# RUN_PREDICTIONS = False
target_outputs = run_all_predictors(target_input, "TARGET")

target_output_summary = pd.DataFrame(
    [
        {
            "predictor": predictor,
            "sample": sample,
            "sequence_type": sequence_type,
            "length": length,
            "file": str(path),
            "bytes": path.stat().st_size,
        }
        for predictor, output_map in target_outputs.items()
        for (sample, sequence_type, length), path in output_map.items()
    ]
)
display(target_output_summary)


Running mixmhcpred: dataset=TARGET, sample=ID8, type=MT, length=8, allele=H2-Kb, n=8,717
Created: /Users/annaklimova/Hussein/predictions/target/mixmhcpred_class_I/mixmhcpred-ID8-MT-H2-Kb-length-8.output.txt (472,290 bytes)
Running mixmhcpred: dataset=TARGET, sample=ID8, type=MT, length=9, allele=H2-Db, n=9,781
Created: /Users/annaklimova/Hussein/predictions/target/mixmhcpred_class_I/mixmhcpred-ID8-MT-H2-Db-length-9.output.txt (540,283 bytes)
Running mixmhcpred: dataset=TARGET, sample=ID8, type=WT, length=8, allele=H2-Kb, n=8,717
Created: /Users/annaklimova/Hussein/predictions/target/mixmhcpred_class_I/mixmhcpred-ID8-WT-H2-Kb-length-8.output.txt (472,456 bytes)
Running mixmhcpred: dataset=TARGET, sample=ID8, type=WT, length=9, allele=H2-Db, n=9,781
Created: /Users/annaklimova/Hussein/predictions/target/mixmhcpred_class_I/mixmhcpred-ID8-WT-H2-Db-length-9.output.txt (540,091 bytes)
Running mixmhcpred: dataset=TARGET, sample=ID8_CPR, type=MT, length=8, allele=H2-Kb, n=8,406
Created: /Users

,predictor,sample,sequence_type,length,file,bytes
0,mixmhcpred,ID8,MT,8,/Users/annaklimova/Hussein/predictions/target/...,472290
1,mixmhcpred,ID8,MT,9,/Users/annaklimova/Hussein/predictions/target/...,540283
2,mixmhcpred,ID8,WT,8,/Users/annaklimova/Hussein/predictions/target/...,472456
3,mixmhcpred,ID8,WT,9,/Users/annaklimova/Hussein/predictions/target/...,540091
4,mixmhcpred,ID8_CPR,MT,8,/Users/annaklimova/Hussein/predictions/target/...,455617
5,mixmhcpred,ID8_CPR,MT,9,/Users/annaklimova/Hussein/predictions/target/...,520371
6,mixmhcpred,ID8_CPR,WT,8,/Users/annaklimova/Hussein/predictions/target/...,455503
7,mixmhcpred,ID8_CPR,WT,9,/Users/annaklimova/Hussein/predictions/target/...,520325
8,netmhcpan,ID8,MT,8,/Users/annaklimova/Hussein/predictions/target/...,781553
9,netmhcpan,ID8,MT,9,/Users/annaklimova/Hussein/predictions/target/...,896278


In [13]:
decoy_outputs = run_all_predictors(decoy_input, "DECOY")

decoy_output_summary = pd.DataFrame(
    [
        {
            "predictor": predictor,
            "sample": sample,
            "sequence_type": sequence_type,
            "length": length,
            "file": str(path),
            "bytes": path.stat().st_size,
        }
        for predictor, output_map in decoy_outputs.items()
        for (sample, sequence_type, length), path in output_map.items()
    ]
)
display(decoy_output_summary)


Running mixmhcpred: dataset=DECOY, sample=DECOY, type=DECOY, length=8, allele=H2-Kb, n=100,000
Created: /Users/annaklimova/Hussein/predictions/decoy/mixmhcpred_class_I/mixmhcpred-DECOY-DECOY-H2-Kb-length-8.output.txt (5,412,083 bytes)
Running mixmhcpred: dataset=DECOY, sample=DECOY, type=DECOY, length=9, allele=H2-Db, n=100,000
Created: /Users/annaklimova/Hussein/predictions/decoy/mixmhcpred_class_I/mixmhcpred-DECOY-DECOY-H2-Db-length-9.output.txt (5,513,481 bytes)
Running netmhcpan: dataset=DECOY, sample=DECOY, type=DECOY, length=8, allele=H-2-Kb, n=100,000
Created: /Users/annaklimova/Hussein/predictions/decoy/netmhcpan_class_I/netmhcpan-DECOY-DECOY-H-2-Kb-length-8.EL-BA.tsv (9,063,907 bytes)
Running netmhcpan: dataset=DECOY, sample=DECOY, type=DECOY, length=9, allele=H-2-Db, n=100,000
Created: /Users/annaklimova/Hussein/predictions/decoy/netmhcpan_class_I/netmhcpan-DECOY-DECOY-H-2-Db-length-9.EL-BA.tsv (9,259,052 bytes)
Running mhcflurry: dataset=DECOY, sample=DECOY, type=DECOY, leng

,predictor,sample,sequence_type,length,file,bytes
0,mixmhcpred,DECOY,DECOY,8,/Users/annaklimova/Hussein/predictions/decoy/m...,5412083
1,mixmhcpred,DECOY,DECOY,9,/Users/annaklimova/Hussein/predictions/decoy/m...,5513481
2,netmhcpan,DECOY,DECOY,8,/Users/annaklimova/Hussein/predictions/decoy/n...,9063907
3,netmhcpan,DECOY,DECOY,9,/Users/annaklimova/Hussein/predictions/decoy/n...,9259052
4,mhcflurry,DECOY,DECOY,8,/Users/annaklimova/Hussein/predictions/decoy/m...,4921183
5,mhcflurry,DECOY,DECOY,9,/Users/annaklimova/Hussein/predictions/decoy/m...,5285423
6,mixmhc2pred,DECOY,DECOY,15,/Users/annaklimova/Hussein/predictions/decoy/m...,6165567
7,netmhciipan,DECOY,DECOY,15,/Users/annaklimova/Hussein/predictions/decoy/n...,10725002
8,hmmhc,DECOY,DECOY,15,/Users/annaklimova/Hussein/predictions/decoy/h...,5274542


In [14]:
target_predictions = combine_all_predictions(target_input, target_outputs)

print(f"Combined TARGET rows: {len(target_predictions):,}")
display(target_predictions.groupby(["sample", "sequence_type", "length"]).size())
display(target_predictions.head())

Combined TARGET rows: 135,538


sample   sequence_type  length
ID8      MT             8          8717
                        9          9781
                        15        16056
         WT             8          8717
                        9          9781
                        15        16056
ID8_CPR  MT             8          8406
                        9          9426
                        15        15383
         WT             8          8406
                        9          9426
                        15        15383
dtype: int64

,sample,sequence_type,peptide,window_row,length,source_row_id,prediction_row_id,mixmhcpred_score,mixmhcpred_rank,netmhcpan_ba_score,netmhcpan_rank,mhcflurry_affinity,mhcflurry_rank,mixmhc2pred_rank,mixmhc2pred_best_allele,netmhciipan_affinity_nM,netmhciipan_rank,hmmhc_score,hmmhc_rank
0,ID8,WT,VKEGKTEK,0,8,ID8|WT|8|0,0,-3.818722,67.314178,0.023269,84.670143,25955.459922,41.740500,NaN,NaN,NaN,NaN,NaN,NaN
1,ID8,WT,KEGKTEKT,1,8,ID8|WT|8|1,1,-3.314452,49.201007,0.011100,96.314262,27407.256137,52.231125,NaN,NaN,NaN,NaN,NaN,NaN
2,ID8,WT,EGKTEKTD,2,8,ID8|WT|8|2,2,-3.598443,59.664842,0.010246,96.597893,28371.192979,59.851250,NaN,NaN,NaN,NaN,NaN,NaN
3,ID8,WT,AVKEGKTEK,3,9,ID8|WT|9|3,0,-2.165814,16.774625,0.013919,64.826744,19945.977559,9.816375,NaN,NaN,NaN,NaN,NaN,NaN
4,ID8,WT,VKEGKTEKT,4,9,ID8|WT|9|4,1,-3.056378,43.995860,0.004879,95.477097,29689.300956,37.885125,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
decoy_predictions = combine_all_predictions(decoy_input, decoy_outputs)
print(f"Combined DECOY rows: {len(decoy_predictions):,}")
display(decoy_predictions.groupby(["sample", "length"]).size())
display(decoy_predictions.head())

Combined DECOY rows: 300,000


sample  length
DECOY   8         100000
        9         100000
        15        100000
dtype: int64

,sample,sequence_type,peptide,allele,length,source_row_id,prediction_row_id,mixmhcpred_score,mixmhcpred_rank,netmhcpan_ba_score,netmhcpan_rank,mhcflurry_affinity,mhcflurry_rank,mixmhc2pred_rank,mixmhc2pred_best_allele,netmhciipan_affinity_nM,netmhciipan_rank,hmmhc_score,hmmhc_rank
0,DECOY,DECOY,QAEELMSN,H2-Kb,8,DECOY|DECOY|8|0,0,-3.230134,46.090330,0.022333,85.952141,28350.702091,59.851250,NaN,NaN,NaN,NaN,NaN,NaN
1,DECOY,DECOY,RASEGVTS,H2-Kb,8,DECOY|DECOY|8|1,1,-3.069141,40.150938,0.032520,71.680733,25891.082934,41.740500,NaN,NaN,NaN,NaN,NaN,NaN
2,DECOY,DECOY,KDSQLIVL,H2-Kb,8,DECOY|DECOY|8|2,2,-1.803245,8.225791,0.072668,33.900703,17365.364773,15.322625,NaN,NaN,NaN,NaN,NaN,NaN
3,DECOY,DECOY,RSTSSTPG,H2-Kb,8,DECOY|DECOY|8|3,3,-3.059795,39.820758,0.057477,44.397568,27267.557686,52.231125,NaN,NaN,NaN,NaN,NaN,NaN
4,DECOY,DECOY,IEEEPAGV,H2-Kb,8,DECOY|DECOY|8|4,4,-2.368899,18.899275,0.039041,63.285652,21272.837642,22.647500,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
target_files = save_predictions_by_sample_and_length(
    target_predictions, "TARGET"
)
display(
    pd.DataFrame(
        {
            "target_file": [str(path) for path in target_files],
            "bytes": [path.stat().st_size for path in target_files],
        }
    )
)


decoy_files = save_predictions_by_sample_and_length(
    decoy_predictions, "DECOY"
)
display(
    pd.DataFrame(
        {
            "decoy_file": [str(path) for path in decoy_files],
            "bytes": [path.stat().st_size for path in decoy_files],
        }
    )
)


create_shared_mutation_outputs()
print(f"Created: {OUTPUT_DIR / 'shared_missense_mutations.tsv'}")
print(f"Created: {OUTPUT_DIR / 'missense_mutations_venn.png'}")

Created: /Users/annaklimova/Hussein/predictions/final_tables/target/ID8.length-8.predictions.tsv (2,102,496 bytes)
Created: /Users/annaklimova/Hussein/predictions/final_tables/target/ID8.length-9.predictions.tsv (2,434,096 bytes)
Created: /Users/annaklimova/Hussein/predictions/final_tables/target/ID8.length-15.predictions.tsv (4,481,204 bytes)
Created: /Users/annaklimova/Hussein/predictions/final_tables/target/ID8_CPR.length-8.predictions.tsv (2,174,273 bytes)
Created: /Users/annaklimova/Hussein/predictions/final_tables/target/ID8_CPR.length-9.predictions.tsv (2,508,130 bytes)
Created: /Users/annaklimova/Hussein/predictions/final_tables/target/ID8_CPR.length-15.predictions.tsv (4,559,289 bytes)


,target_file,bytes
0,/Users/annaklimova/Hussein/predictions/final_t...,2102496
1,/Users/annaklimova/Hussein/predictions/final_t...,2434096
2,/Users/annaklimova/Hussein/predictions/final_t...,4481204
3,/Users/annaklimova/Hussein/predictions/final_t...,2174273
4,/Users/annaklimova/Hussein/predictions/final_t...,2508130
5,/Users/annaklimova/Hussein/predictions/final_t...,4559289


Created: /Users/annaklimova/Hussein/predictions/final_tables/decoy/DECOY.length-8.predictions.tsv (13,206,694 bytes)
Created: /Users/annaklimova/Hussein/predictions/final_tables/decoy/DECOY.length-9.predictions.tsv (13,680,918 bytes)
Created: /Users/annaklimova/Hussein/predictions/final_tables/decoy/DECOY.length-15.predictions.tsv (15,063,458 bytes)


,decoy_file,bytes
0,/Users/annaklimova/Hussein/predictions/final_t...,13206694
1,/Users/annaklimova/Hussein/predictions/final_t...,13680918
2,/Users/annaklimova/Hussein/predictions/final_t...,15063458


Created: /Users/annaklimova/Hussein/predictions/shared_missense_mutations.tsv
Created: /Users/annaklimova/Hussein/predictions/missense_mutations_venn.png
